# OCR Gate 2 — dev-subset-5 evaluation (no Gemini calls)

Attach `thvu165/aic-2026-keyframes`, select **GPU T4 x2**, enable Internet, and Run All. This dev-only harness masks GPU 1 and processes all 4,164 keyframes from `L21_V001`, `L21_V002`, `L21_V003`, `L21_V005`, and `L21_V006`.

It verifies catalog UID-set digests, pinned CRAFT/latin_g2/Vintern bytes, runs CRAFT+EasyOCR on every frame, routes seed escalation candidates to Vintern FP16, measures throughput/ETA, and exports a 250-region manual-review sample. It does **not** call Gemini, change `OcrResult`, publish production artifacts, or mark Gate 2 accuracy PASS automatically.

JSONL files under `/kaggle/working/ocr-gate2-dev5-v1` are authoritative checkpoints. Rerunning the notebook in the same Kaggle session resumes by UID/candidate ID without duplicates.

In [ ]:
# Keep first: use only GPU 0 and preserve Kaggle's NumPy/OpenCV/Torch ABI.
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import importlib.metadata as md
import importlib.util
import subprocess
import sys

def version_or_missing(name):
    try:
        return md.version(name)
    except md.PackageNotFoundError:
        return None

protected_names = ["torch", "torchvision", "numpy", "opencv-python", "opencv-python-headless", "scikit-image", "pandas"]
protected_before = {name: version_or_missing(name) for name in protected_names}
python_only_runtime = [
    "transformers==4.43.4", "tokenizers==0.19.1", "huggingface-hub==0.24.7",
    "easyocr==1.7.2", "python-bidi==0.6.6", "pyclipper==1.3.0.post6",
    "ninja==1.11.1.4", "sentencepiece==0.2.0",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--no-cache-dir", "--no-deps", *python_only_runtime])
required_modules = ["accelerate", "safetensors", "timm", "einops", "yaml", "scipy", "skimage", "shapely"]
missing = [name for name in required_modules if importlib.util.find_spec(name) is None]
assert not missing, ("Kaggle image missing expected modules", missing)
protected_after = {name: version_or_missing(name) for name in protected_names}
assert protected_before == protected_after, ("pip changed Kaggle binary packages", protected_before, protected_after)
abi = subprocess.run([sys.executable, "-c", "import numpy,cv2,pandas,skimage; print(numpy.__version__,cv2.__version__,pandas.__version__,skimage.__version__)"], text=True, capture_output=True)
assert abi.returncode == 0, "Dirty binary ABI; stop the Kaggle session completely and start fresh.\n" + abi.stderr
print("ABI_PROBE", abi.stdout.strip())
print({"protected": protected_after, "transformers": version_or_missing("transformers"), "easyocr": version_or_missing("easyocr")})

In [ ]:
from __future__ import annotations

import csv
import gc
import hashlib
import json
import math
import platform
import re
import shutil
import statistics
import time
import unicodedata
import urllib.request
import zipfile
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path

import cv2
import numpy as np
import torch
from PIL import Image, ImageDraw, ImageFont

DEV_EXPECTED = {
    "L21_V001": {"count": 1008, "uid_set_sha256": "d97f6d1cb014354b11942ea908b2f75fb7ec423dc44b87451b8fe157fc16eee2"},
    "L21_V002": {"count": 843, "uid_set_sha256": "62773f4ecc75cac52d6df5f598734fd9f608afa3ea680a74da91b983031596ca"},
    "L21_V003": {"count": 765, "uid_set_sha256": "b7d1fe33ecfeb644506b6015ee98a6e03b9b8a9baade275fab303cbe7bd4d03b"},
    "L21_V005": {"count": 744, "uid_set_sha256": "2f16c1ddc51b87bfe3299f99910f01bf7e2b8ec718b6f99015cdda21982199b0"},
    "L21_V006": {"count": 804, "uid_set_sha256": "b3b6e61ff01dbdef3afc2de96f8a077c992f43e32640264743b707bfe84942cc"},
}
CATALOG_SHA256 = "ee9693e75580527a0a257e9ba003984e105b059b716922c03c7a0b72b1508a37"
CATALOG_RECORD_COUNT = 293_336
EASY_CONF_ESCALATE = 0.60
WIDE_CROP_ASPECT_FOR_TWO_TILES = 4.0
MANUAL_REVIEW_TARGET = 250
RESUME_SEED_RECORDS = 50

MODEL_ID = "5CD-AI/Vintern-1B-v3_5"
MODEL_REVISION = "b98f263eab246eb5269ade64edbdca8a887dc44d"
MODEL_WEIGHT_BYTES = 3_752_849_256
MODEL_WEIGHT_SHA256 = "296a16a6bf28e6d3f0fb9298deba70b3cfa1d7519f4aa326e2f862bf2e63be05"
MODEL_REQUIRED_GIT_OIDS = {
    "config.json": "2668519f652eddcc2abbb56a52518d38c4f88887",
    "configuration_intern_vit.py": "7e630c456eb9cf350e55bf850c3ff72f445a7e17",
    "configuration_internvl_chat.py": "799209432caf749e77de4c889ec03fe6a32fcdf9",
    "conversation.py": "76fcea7f331de42b6aed7e39fdf80728f9784b7f",
    "modeling_intern_vit.py": "1c5c043a4b860720b3b6e55107e8e6ecf0c573de",
    "modeling_internvl_chat.py": "41f48cd5cb907e025725fc42ac819cf3f03c01b5",
}
EASYOCR_WEIGHTS = {
    "craft_mlt_25k": {"url": "https://github.com/JaidedAI/EasyOCR/releases/download/pre-v1.1.6/craft_mlt_25k.zip", "zip_sha256": "8dc6a1c703a89ed56308ef742d26ebd45c656248cbbbda6e7fe60e569f873e65", "weight_name": "craft_mlt_25k.pth", "weight_sha256": "4a5efbfb48b4081100544e75e1e2b57f8de3d84f213004b14b85fd4b3748db17"},
    "latin_g2": {"url": "https://github.com/JaidedAI/EasyOCR/releases/download/v1.3/latin_g2.zip", "zip_sha256": "29f1920c493378da65a59793fb70e7e190504662b6bed57ab26f4067eb5f3769", "weight_name": "latin_g2.pth", "weight_sha256": "aaa95be1c4a9cb3496879bed7c520886ce1164f89e026f0c54488394e74e8c55"},
}

INPUT_ROOT = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working/ocr-gate2-dev5-v1")
MODEL_DIR = WORK_DIR / "easyocr-models"
REGION_DIR = WORK_DIR / "region-crops"
SHEET_DIR = WORK_DIR / "manual-review-sheets"
EASY_JSONL = WORK_DIR / "easyocr-frames.jsonl"
CANDIDATE_JSONL = WORK_DIR / "vintern-candidates.jsonl"
VINTERN_JSONL = WORK_DIR / "vintern-results.jsonl"
GEMINI_RESIDUAL_JSONL = Path("/kaggle/working/ocr_gate2_gemini_residual_canary.jsonl")
REVIEW_CSV = Path("/kaggle/working/ocr_gate2_manual_review.csv")
REVIEW_ZIP_BASE = Path("/kaggle/working/ocr_gate2_manual_review_sheets")
REPORT_JSON = Path("/kaggle/working/ocr_gate2_dev5_report.json")
for path in (WORK_DIR, MODEL_DIR, REGION_DIR, SHEET_DIR): path.mkdir(parents=True, exist_ok=True)

def utc_now(): return datetime.now(timezone.utc).isoformat()
def sha256_file(path, chunk_size=8*1024*1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while chunk := handle.read(chunk_size): digest.update(chunk)
    return digest.hexdigest()
def git_blob_oid(path, chunk_size=8*1024*1024):
    path = Path(path); digest = hashlib.sha1(); digest.update(f"blob {path.stat().st_size}\0".encode("ascii"))
    with path.open("rb") as handle:
        while chunk := handle.read(chunk_size): digest.update(chunk)
    return digest.hexdigest()
def make_uid(video_id, shot_id, local_idx):
    raw = f"{video_id}:{shot_id}:{local_idx}".encode()
    return int.from_bytes(hashlib.blake2b(raw, digest_size=8).digest(), "big", signed=False) >> 1
def uid_set_sha256(values): return hashlib.sha256("".join(f"{value}\n" for value in sorted(values)).encode()).hexdigest()
def stable_id(*parts): return hashlib.sha256("|".join(map(str, parts)).encode()).hexdigest()[:24]
def atomic_json(path, value):
    path = Path(path); temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8"); os.replace(temporary, path)
def rewrite_jsonl(path, records):
    path = Path(path); temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8") as handle:
        for record in records: handle.write(json.dumps(record, ensure_ascii=False, sort_keys=True) + "\n")
    os.replace(temporary, path)
def append_jsonl(path, record):
    with Path(path).open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(record, ensure_ascii=False, sort_keys=True) + "\n"); handle.flush(); os.fsync(handle.fileno())
def load_jsonl(path, id_field):
    rows=[]; seen=set(); path=Path(path)
    if not path.exists(): return rows
    for line_number, line in enumerate(path.read_text(encoding="utf-8").splitlines(), 1):
        if not line.strip(): continue
        row=json.loads(line); identity=row[id_field]
        if identity in seen: raise ValueError(f"duplicate {id_field} at line {line_number}: {identity}")
        seen.add(identity); rows.append(row)
    return rows
def package_versions():
    return {name: version_or_missing(name) for name in ["torch","torchvision","transformers","easyocr","accelerate","safetensors","timm","einops","opencv-python","numpy","Pillow"]}
def gpu_query():
    output=subprocess.check_output(["nvidia-smi","--query-gpu=index,name,memory.total,memory.used,driver_version","--format=csv,noheader,nounits"],text=True)
    return [line.strip() for line in output.splitlines() if line.strip()]
assert torch.cuda.is_available() and torch.cuda.device_count()==1
assert "T4" in torch.cuda.get_device_name(0).upper(), torch.cuda.get_device_name(0)
print("GPU", torch.cuda.get_device_name(0), gpu_query())

In [ ]:
# Discover exactly the five video directories and validate their UID sets against frames.csv evidence.
filename_pattern = re.compile(r"^(s\d+)_(\d+)\.(?:jpg|jpeg|png|webp)$", re.IGNORECASE)
catalog_rows = []
catalog_validation = {}
for video_id, expected in DEV_EXPECTED.items():
    matches = [path for path in INPUT_ROOT.glob(f"**/keyframes-batch-*/{video_id}") if path.is_dir()]
    assert len(matches) == 1, (video_id, [str(path) for path in matches])
    video_dir = matches[0]
    image_entries = []
    for image_path in sorted(video_dir.iterdir()):
        match = filename_pattern.match(image_path.name)
        if not match: continue
        shot_id, filename_index_text = match.groups()
        image_entries.append((shot_id, int(filename_index_text), image_path))
    image_entries.sort(key=lambda item: (item[0], item[1]))
    rows = []
    for canonical_local_idx, (shot_id, filename_index, image_path) in enumerate(image_entries):
        rows.append({"video_id": video_id, "shot_id": shot_id, "local_idx": canonical_local_idx, "filename_index": filename_index, "keyframe_uid": make_uid(video_id, shot_id, canonical_local_idx), "source_image": image_path.relative_to(INPUT_ROOT).as_posix(), "source_path": str(image_path)})
    uid_values = [row["keyframe_uid"] for row in rows]
    actual_digest = uid_set_sha256(uid_values)
    assert len(rows) == expected["count"], (video_id, len(rows), expected["count"])
    assert len(uid_values) == len(set(uid_values)), f"duplicate keyframe_uid in {video_id}"
    assert actual_digest == expected["uid_set_sha256"], (video_id, actual_digest, expected["uid_set_sha256"])
    catalog_validation[video_id] = {"count": len(rows), "uid_set_sha256": actual_digest, "directory": str(video_dir)}
    catalog_rows.extend(rows)
catalog_rows.sort(key=lambda row: (row["video_id"], row["shot_id"], row["local_idx"]))
assert len(catalog_rows) == 4_164
assert len({row["keyframe_uid"] for row in catalog_rows}) == len(catalog_rows)
print(json.dumps(catalog_validation, indent=2)); print("DEV_TOTAL", len(catalog_rows))

In [ ]:
# Pinned EasyOCR CRAFT + latin_g2, full dev5 pass with append-safe UID resume.
import easyocr

def install_verified_easyocr_weights():
    verified = {}
    for key, spec in EASYOCR_WEIGHTS.items():
        archive_path = WORK_DIR / f"{key}.zip"; weight_path = MODEL_DIR / spec["weight_name"]
        if not archive_path.exists() or sha256_file(archive_path) != spec["zip_sha256"]: urllib.request.urlretrieve(spec["url"], archive_path)
        assert sha256_file(archive_path) == spec["zip_sha256"], f"{key} ZIP checksum mismatch"
        if not weight_path.exists() or sha256_file(weight_path) != spec["weight_sha256"]:
            with zipfile.ZipFile(archive_path) as archive:
                member = next(name for name in archive.namelist() if name.endswith(spec["weight_name"]))
                with archive.open(member) as source, weight_path.open("wb") as target: shutil.copyfileobj(source, target)
        assert sha256_file(weight_path) == spec["weight_sha256"], f"{key} weight checksum mismatch"
        verified[key] = {"zip_sha256": spec["zip_sha256"], "weight_sha256": spec["weight_sha256"]}
    return verified

easyocr_weight_provenance = install_verified_easyocr_weights()
def new_reader():
    return easyocr.Reader(["vi","en"], gpu="cuda:0", model_storage_directory=str(MODEL_DIR), download_enabled=False, detector=True, recognizer=True, verbose=True)
def vietnamese_marked(text):
    base = "ăâđêôơưĂÂĐÊÔƠƯ"
    return any(char in base or (unicodedata.category(char)=="Ll" and any(mark in unicodedata.normalize("NFD", char) for mark in "\u0300\u0301\u0303\u0309\u0323")) for char in text)
def alphabetic_ascii(text): return bool(re.search(r"[A-Za-z]{3,}", text))
def quad_flat(points): return [float(value) for point in points for value in point]
def normalize_quad(points, width, height): return [max(0.0,min(1.0,float(value)/(width if index%2==0 else height))) for index,value in enumerate(quad_flat(points))]

def process_easyocr_frame(reader, row):
    started=time.perf_counter(); image=cv2.imread(row["source_path"],cv2.IMREAD_COLOR)
    if image is None: return {**row,"schema_version":1,"status":"error","latency_seconds":time.perf_counter()-started,"error":"cv2_imread_failed","regions":[]}
    height,width=image.shape[:2]
    try:
        detect_started=time.perf_counter()
        horizontal,free=reader.detect(image,min_size=10,text_threshold=0.6,low_text=0.3,link_threshold=0.3,canvas_size=2560,mag_ratio=1.0,slope_ths=0.1,ycenter_ths=0.5,height_ths=0.5,width_ths=0.5,add_margin=0.1,reformat=True)
        detect_seconds=time.perf_counter()-detect_started; horizontal0=horizontal[0] if horizontal else []; free0=free[0] if free else []; detected_count=len(horizontal0)+len(free0)
        if detected_count==0:
            return {**row,"schema_version":1,"status":"no_text","image_width":width,"image_height":height,"detected_region_count":0,"detect_seconds":detect_seconds,"recognize_seconds":0.0,"latency_seconds":time.perf_counter()-started,"error":None,"frame_mixed_candidate":False,"regions":[]}
        grey=cv2.cvtColor(image,cv2.COLOR_BGR2GRAY); recognize_started=time.perf_counter()
        results=reader.recognize(grey,horizontal0,free0,decoder="greedy",beamWidth=5,batch_size=1,workers=0,detail=1,paragraph=False,contrast_ths=0.1,adjust_contrast=0.5,filter_ths=0.003,reformat=False)
        recognize_seconds=time.perf_counter()-recognize_started
        if len(results)!=detected_count: raise RuntimeError(f"recognizer returned {len(results)} rows for {detected_count} CRAFT regions")
        parsed=[]
        for index,(points,text_value,confidence_value) in enumerate(results):
            points=np.asarray(points,dtype=np.float32).reshape(4,2); text_value=str(text_value).strip(); confidence=float(max(0.0,min(1.0,confidence_value)))
            x1=max(0,int(math.floor(points[:,0].min()))-4); y1=max(0,int(math.floor(points[:,1].min()))-4); x2=min(width,int(math.ceil(points[:,0].max()))+4); y2=min(height,int(math.ceil(points[:,1].max()))+4)
            region_id=stable_id(row["keyframe_uid"],index,";".join(f"{v:.2f}" for v in quad_flat(points)))
            crop_path=REGION_DIR/f"{region_id}.jpg"
            if not crop_path.exists() and x2>x1 and y2>y1: assert cv2.imwrite(str(crop_path),image[y1:y2,x1:x2])
            parsed.append({"region_id":region_id,"bbox_px":quad_flat(points),"bbox_normalized":normalize_quad(points,width,height),"crop_path":str(crop_path),"crop_width":x2-x1,"crop_height":y2-y1,"easyocr_text":text_value,"easyocr_confidence":confidence,"has_vi_marks":vietnamese_marked(text_value),"has_ascii_word":alphabetic_ascii(text_value)})
        frame_mixed = any(region["has_vi_marks"] for region in parsed) and any(region["has_ascii_word"] and not region["has_vi_marks"] for region in parsed)
        for region in parsed:
            reasons=[]
            if not region["easyocr_text"]: reasons.append("empty")
            if region["easyocr_confidence"] < EASY_CONF_ESCALATE: reasons.append("low_confidence")
            if frame_mixed: reasons.append("mixed_frame")
            region["escalation_reasons"]=reasons
        return {**row,"schema_version":1,"status":"text_detected","image_width":width,"image_height":height,"detected_region_count":detected_count,"detect_seconds":detect_seconds,"recognize_seconds":recognize_seconds,"latency_seconds":time.perf_counter()-started,"error":None,"frame_mixed_candidate":frame_mixed,"regions":parsed}
    except Exception as exc:
        return {**row,"schema_version":1,"status":"error","image_width":width,"image_height":height,"detected_region_count":None,"detect_seconds":None,"recognize_seconds":None,"latency_seconds":time.perf_counter()-started,"error":f"{type(exc).__name__}: {str(exc)[:500]}","regions":[]}

def run_easyocr_pass(reader,max_new=None):
    existing=load_jsonl(EASY_JSONL,"keyframe_uid"); done={row["keyframe_uid"] for row in existing}; new_count=0
    if max_new == 0: return {"preexisting":len(done),"new":0,"after":len(done)}
    for row in catalog_rows:
        if row["keyframe_uid"] in done: continue
        append_jsonl(EASY_JSONL,process_easyocr_frame(reader,row)); new_count+=1
        if new_count%50==0: print("EASYOCR_PROGRESS",len(done)+new_count,"/",len(catalog_rows))
        if max_new is not None and new_count>=max_new: break
    return {"preexisting":len(done),"new":new_count,"after":len(done)+new_count}

initial_existing=len(load_jsonl(EASY_JSONL,"keyframe_uid"))
reader=new_reader(); seed=run_easyocr_pass(reader,max_new=max(0,RESUME_SEED_RECORDS-initial_existing))
del reader; gc.collect(); torch.cuda.empty_cache()
resume_preexisting=len(load_jsonl(EASY_JSONL,"keyframe_uid"))
reader=new_reader(); resumed=run_easyocr_pass(reader,max_new=None)
del reader; gc.collect(); torch.cuda.empty_cache()
easy_rows=load_jsonl(EASY_JSONL,"keyframe_uid")
assert len(easy_rows)==len(catalog_rows)==4_164
assert {row["keyframe_uid"] for row in easy_rows}=={row["keyframe_uid"] for row in catalog_rows}
resume_demo={"initial_existing":initial_existing,"seed_pass":seed,"model_reloaded_with_preexisting":resume_preexisting,"resume_pass":resumed,"final_records":len(easy_rows),"duplicates":0}
print("RESUME_DEMO",json.dumps(resume_demo,indent=2)); print("EASYOCR_STATUS",Counter(row["status"] for row in easy_rows))

In [ ]:
# Build deterministic seed escalation list and a confidence-threshold sweep.
easy_rows=load_jsonl(EASY_JSONL,"keyframe_uid")
candidate_rows=[]
for frame in easy_rows:
    for region in frame.get("regions",[]):
        if region["escalation_reasons"]:
            candidate_rows.append({"schema_version":1,"candidate_id":region["region_id"],"video_id":frame["video_id"],"keyframe_uid":frame["keyframe_uid"],"shot_id":frame["shot_id"],"local_idx":frame["local_idx"],"source_image":frame["source_image"],"crop_path":region["crop_path"],"crop_width":region["crop_width"],"crop_height":region["crop_height"],"easyocr_text":region["easyocr_text"],"easyocr_confidence":region["easyocr_confidence"],"escalation_reasons":region["escalation_reasons"]})
candidate_rows.sort(key=lambda row:(row["video_id"],row["keyframe_uid"],row["candidate_id"]))
assert len({row["candidate_id"] for row in candidate_rows})==len(candidate_rows)
rewrite_jsonl(CANDIDATE_JSONL,candidate_rows)
all_regions=[region for frame in easy_rows for region in frame.get("regions",[])]
threshold_sweep=[]
for threshold in (0.30,0.40,0.50,0.60,0.70,0.80):
    count=sum(1 for region in all_regions if not region["easyocr_text"] or region["easyocr_confidence"]<threshold or "mixed_frame" in region["escalation_reasons"])
    threshold_sweep.append({"confidence_threshold":threshold,"candidate_regions":count,"candidate_fraction":count/max(1,len(all_regions))})
print("REGIONS",len(all_regions),"SEED_CANDIDATES",len(candidate_rows)); print(json.dumps(threshold_sweep,indent=2))

In [ ]:
# Load the exact official Vintern revision as FP16 using a verified local runtime view.
from huggingface_hub import snapshot_download
from transformers import AutoModel, AutoTokenizer

snapshot_started=time.perf_counter(); snapshot_dir=Path(snapshot_download(repo_id=MODEL_ID,revision=MODEL_REVISION)); snapshot_seconds=time.perf_counter()-snapshot_started
weight_path=snapshot_dir/"model.safetensors"
assert weight_path.stat().st_size==MODEL_WEIGHT_BYTES and sha256_file(weight_path)==MODEL_WEIGHT_SHA256
actual_code_oids={}
for filename,expected in MODEL_REQUIRED_GIT_OIDS.items():
    source=snapshot_dir/filename; assert source.is_file(); actual=git_blob_oid(source); assert actual==expected,(filename,actual,expected); actual_code_oids[filename]=actual
runtime_dir=WORK_DIR/"vintern-local-runtime"
if runtime_dir.exists(): shutil.rmtree(runtime_dir)
runtime_dir.mkdir()
for source in snapshot_dir.iterdir():
    target=runtime_dir/source.name
    if source.name=="config.json":
        config=json.loads(source.read_text(encoding="utf-8")); config["auto_map"]={key:value.split("--",1)[-1] for key,value in config["auto_map"].items()}; target.write_text(json.dumps(config,indent=2),encoding="utf-8")
    elif source.is_file(): target.symlink_to(source.resolve())
torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(0); free_before,total_bytes=torch.cuda.mem_get_info(0); load_started=time.perf_counter()
tokenizer=AutoTokenizer.from_pretrained(str(runtime_dir),trust_remote_code=True,use_fast=False,local_files_only=True)
model=AutoModel.from_pretrained(str(runtime_dir),torch_dtype=torch.float16,low_cpu_mem_usage=True,trust_remote_code=True,local_files_only=True,use_flash_attn=False).eval().to("cuda:0")
torch.cuda.synchronize(); load_seconds=time.perf_counter()-load_started; free_after,_=torch.cuda.mem_get_info(0)
vintern_load={"seconds":load_seconds,"used_delta_mib":(free_before-free_after)/2**20,"allocated_mib":torch.cuda.memory_allocated(0)/2**20,"peak_allocated_mib":torch.cuda.max_memory_allocated(0)/2**20}
assert all(parameter.dtype==torch.float16 for parameter in model.parameters() if parameter.is_floating_point())
print("VINTERN_LOAD",vintern_load)

In [ ]:
# Vintern preprocessing and resumable inference over every seed escalation candidate.
import torchvision.transforms as T
from torchvision.transforms.functional import InterpolationMode
IMAGENET_MEAN=(0.485,0.456,0.406); IMAGENET_STD=(0.229,0.224,0.225)
transform=T.Compose([T.Lambda(lambda image:image.convert("RGB")),T.Resize((448,448),interpolation=InterpolationMode.BICUBIC),T.ToTensor(),T.Normalize(mean=IMAGENET_MEAN,std=IMAGENET_STD)])
def closest_ratio(aspect,target_ratios,width,height,image_size):
    best=(1,1); diff=float("inf"); area=width*height
    for ratio in target_ratios:
        current=abs(aspect-ratio[0]/ratio[1])
        if current<diff or (current==diff and area>0.5*image_size*image_size*ratio[0]*ratio[1]): diff=current; best=ratio
    return best
def dynamic_tiles(image,max_num,image_size=448):
    width,height=image.size; ratios=sorted({(i,j) for n in range(1,max_num+1) for i in range(1,n+1) for j in range(1,n+1) if 1<=i*j<=max_num},key=lambda pair:pair[0]*pair[1]); ratio=closest_ratio(width/height,ratios,width,height,image_size)
    target_width=image_size*ratio[0]; target_height=image_size*ratio[1]; resized=image.resize((target_width,target_height)); images=[]
    for index in range(ratio[0]*ratio[1]):
        columns=target_width//image_size; box=((index%columns)*image_size,(index//columns)*image_size,((index%columns)+1)*image_size,((index//columns)+1)*image_size); images.append(resized.crop(box))
    if len(images)!=1: images.append(image.resize((image_size,image_size)))
    return images
def load_pixels(path,max_num):
    tiles=dynamic_tiles(Image.open(path).convert("RGB"),max_num=max_num); return torch.stack([transform(tile) for tile in tiles]).to("cuda:0",dtype=torch.float16),len(tiles)
def normalized_similarity(left,right):
    left=unicodedata.normalize("NFC",left).casefold().strip(); right=unicodedata.normalize("NFC",right).casefold().strip()
    if not left and not right:return 1.0
    previous=list(range(len(right)+1))
    for i,char_left in enumerate(left,1):
        current=[i]
        for j,char_right in enumerate(right,1): current.append(min(current[-1]+1,previous[j]+1,previous[j-1]+(char_left!=char_right)))
        previous=current
    return 1.0-previous[-1]/max(1,len(left),len(right))
QUESTION="<image>\nChép lại nguyên văn toàn bộ chữ nhìn thấy. Không sửa chính tả, không suy đoán phần bị che. Chỉ trả về văn bản."
GENERATION={"max_new_tokens":96,"do_sample":False,"num_beams":1,"eos_token_id":tokenizer.eos_token_id,"pad_token_id":tokenizer.eos_token_id}

candidate_rows=load_jsonl(CANDIDATE_JSONL,"candidate_id"); prior=load_jsonl(VINTERN_JSONL,"candidate_id"); done={row["candidate_id"] for row in prior}
warm_candidate=next((row for row in candidate_rows if row["candidate_id"] not in done),None)
if warm_candidate:
    warm_pixels,_=load_pixels(warm_candidate["crop_path"],1)
    with torch.inference_mode(): model.chat(tokenizer,warm_pixels,QUESTION,GENERATION)
    del warm_pixels; torch.cuda.synchronize()
for index,candidate in enumerate(candidate_rows):
    if candidate["candidate_id"] in done: continue
    max_num=2 if candidate["crop_height"]>0 and candidate["crop_width"]/candidate["crop_height"]>=WIDE_CROP_ASPECT_FOR_TWO_TILES else 1
    started=time.perf_counter()
    try:
        pixels,tile_count=load_pixels(candidate["crop_path"],max_num); torch.cuda.synchronize(); infer_started=time.perf_counter()
        with torch.inference_mode(): text_value=str(model.chat(tokenizer,pixels,QUESTION,GENERATION)).strip()
        torch.cuda.synchronize(); inference_seconds=time.perf_counter()-infer_started; del pixels
        similarity=normalized_similarity(candidate["easyocr_text"],text_value); residual=[]
        if not text_value: residual.append("vintern_empty")
        if candidate["easyocr_text"] and candidate["easyocr_confidence"]>=0.45 and similarity<0.30: residual.append("strong_disagreement")
        record={**candidate,"status":"success" if text_value else "empty","vintern_text":text_value,"max_num":max_num,"tile_count":tile_count,"inference_seconds":inference_seconds,"total_seconds":time.perf_counter()-started,"text_similarity":similarity,"gemini_residual_reasons":residual,"error":None}
    except Exception as exc:
        if isinstance(exc,torch.cuda.OutOfMemoryError): torch.cuda.empty_cache()
        record={**candidate,"status":"error","vintern_text":"","max_num":max_num,"tile_count":None,"inference_seconds":None,"total_seconds":time.perf_counter()-started,"text_similarity":None,"gemini_residual_reasons":["vintern_error"],"error":f"{type(exc).__name__}: {str(exc)[:500]}"}
    append_jsonl(VINTERN_JSONL,record)
    if (index+1)%50==0: print("VINTERN_PROGRESS",index+1,"/",len(candidate_rows))
vintern_rows=load_jsonl(VINTERN_JSONL,"candidate_id")
assert {row["candidate_id"] for row in vintern_rows}=={row["candidate_id"] for row in candidate_rows}
vintern_peak={"allocated_mib":torch.cuda.max_memory_allocated(0)/2**20,"reserved_mib":torch.cuda.max_memory_reserved(0)/2**20}
print("VINTERN_STATUS",Counter(row["status"] for row in vintern_rows),vintern_peak)
del model,tokenizer; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Report, Gemini residual canary list, and deterministic 250-region manual review package.
easy_rows=load_jsonl(EASY_JSONL,"keyframe_uid"); vintern_rows=load_jsonl(VINTERN_JSONL,"candidate_id"); vintern_by_id={row["candidate_id"]:row for row in vintern_rows}
residual_rows=[row for row in vintern_rows if row["gemini_residual_reasons"]]
residual_rows.sort(key=lambda row:(row["status"]!="error",row["text_similarity"] if row["text_similarity"] is not None else -1,row["candidate_id"]))
rewrite_jsonl(GEMINI_RESIDUAL_JSONL,residual_rows[:100])
region_pool=[]
for frame in easy_rows:
    for region in frame.get("regions",[]):
        vintern=vintern_by_id.get(region["region_id"]); route="easyocr_pass" if vintern is None else ("gemini_residual" if vintern["gemini_residual_reasons"] else "vintern_pass")
        region_pool.append({"review_id":stable_id("review",region["region_id"]),"video_id":frame["video_id"],"keyframe_uid":frame["keyframe_uid"],"source_image":frame["source_image"],"region_id":region["region_id"],"crop_path":region["crop_path"],"route":route,"escalation_reasons":"|".join(region["escalation_reasons"]),"easyocr_text":region["easyocr_text"],"easyocr_confidence":region["easyocr_confidence"],"vintern_text":vintern["vintern_text"] if vintern else "","human_text":"","craft_correct":"","easyocr_correct":"","vintern_correct":"","preferred_engine":""})
    if frame["status"]=="no_text":
        region_pool.append({"review_id":stable_id("review-no-text",frame["keyframe_uid"]),"video_id":frame["video_id"],"keyframe_uid":frame["keyframe_uid"],"source_image":frame["source_image"],"region_id":"","crop_path":frame["source_path"],"route":"craft_no_text_control","escalation_reasons":"","easyocr_text":"","easyocr_confidence":"","vintern_text":"","human_text":"","craft_correct":"","easyocr_correct":"","vintern_correct":"","preferred_engine":""})
review=[]
for video_id in DEV_EXPECTED:
    video_rows=[row for row in region_pool if row["video_id"]==video_id]
    chosen=[]
    for route,quota in (("craft_no_text_control",10),("gemini_residual",10),("vintern_pass",15),("easyocr_pass",15)):
        pool=sorted((row for row in video_rows if row["route"]==route),key=lambda row:hashlib.sha256(row["review_id"].encode()).hexdigest()); chosen.extend(pool[:quota])
    chosen_ids={row["review_id"] for row in chosen}; fill=sorted((row for row in video_rows if row["review_id"] not in chosen_ids),key=lambda row:hashlib.sha256(("fill"+row["review_id"]).encode()).hexdigest()); chosen.extend(fill[:max(0,50-len(chosen))]); review.extend(chosen[:50])
assert len({row["review_id"] for row in review})==len(review) and len(review)<=MANUAL_REVIEW_TARGET
fieldnames=list(review[0].keys()) if review else []
with REVIEW_CSV.open("w",encoding="utf-8-sig",newline="") as handle:
    writer=csv.DictWriter(handle,fieldnames=fieldnames); writer.writeheader(); writer.writerows(review)
if SHEET_DIR.exists(): shutil.rmtree(SHEET_DIR)
SHEET_DIR.mkdir(parents=True)
font_path=Path("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"); font=ImageFont.truetype(str(font_path),14) if font_path.exists() else ImageFont.load_default()
for sheet_index in range(0,len(review),16):
    subset=review[sheet_index:sheet_index+16]; canvas=Image.new("RGB",(1600,1200),"white"); draw=ImageDraw.Draw(canvas)
    for offset,row in enumerate(subset):
        column=offset%4; line=offset//4; x=column*400; y=line*300
        crop=Image.open(row["crop_path"]).convert("RGB"); crop.thumbnail((380,180)); canvas.paste(crop,(x+10,y+5))
        confidence=f"{row['easyocr_confidence']:.2f}" if isinstance(row["easyocr_confidence"],(int,float)) else "-"
        label=f"{row['video_id']} {row['route']}\nE:{row['easyocr_text'][:38]} ({confidence})\nV:{row['vintern_text'][:38]}"; draw.multiline_text((x+10,y+195),label,fill="black",font=font,spacing=2)
    canvas.save(SHEET_DIR/f"review-{sheet_index//16:02d}.jpg",quality=92)
review_zip=Path(shutil.make_archive(str(REVIEW_ZIP_BASE),"zip",root_dir=SHEET_DIR))

all_regions=[region for frame in easy_rows for region in frame.get("regions",[])]
easy_seconds=sum(row["latency_seconds"] for row in easy_rows); craft_seconds=sum((row.get("detect_seconds") or 0.0) for row in easy_rows); recognize_seconds=sum((row.get("recognize_seconds") or 0.0) for row in easy_rows); vintern_success=[row for row in vintern_rows if row["status"]=="success"]; vintern_seconds=sum(row["inference_seconds"] for row in vintern_success)
by_video={}
for video_id in DEV_EXPECTED:
    frames=[row for row in easy_rows if row["video_id"]==video_id]; regions=[region for frame in frames for region in frame.get("regions",[])]; candidate_ids={region["region_id"] for region in regions if region["escalation_reasons"]}; vrows=[row for row in vintern_rows if row["candidate_id"] in candidate_ids]
    by_video[video_id]={"frames":len(frames),"no_text_frames":sum(row["status"]=="no_text" for row in frames),"error_frames":sum(row["status"]=="error" for row in frames),"regions":len(regions),"vintern_candidates":len(candidate_ids),"candidate_fraction":len(candidate_ids)/max(1,len(regions)),"vintern_errors":sum(row["status"]=="error" for row in vrows),"gemini_residuals":sum(bool(row["gemini_residual_reasons"]) for row in vrows)}
easy_hours_full=(easy_seconds/len(easy_rows))*CATALOG_RECORD_COUNT/3600; candidates_per_frame=len(vintern_rows)/len(easy_rows); vintern_seconds_per_candidate=vintern_seconds/max(1,len(vintern_success)); vintern_hours_full=candidates_per_frame*CATALOG_RECORD_COUNT*vintern_seconds_per_candidate/3600
errors=sum(row["status"]=="error" for row in easy_rows)+sum(row["status"]=="error" for row in vintern_rows)
report={"schema_version":1,"created_utc":utc_now(),"decision":"PENDING_MANUAL_ACCURACY" if errors==0 else "FAIL_RUNTIME_ERRORS","scope":"dev_only_no_gemini_calls","catalog":{"full_catalog_sha256":CATALOG_SHA256,"full_catalog_records":CATALOG_RECORD_COUNT,"dev_records":len(easy_rows),"validation":catalog_validation},"environment":{"python":sys.version,"platform":platform.platform(),"gpu":torch.cuda.get_device_name(0),"packages":package_versions()},"provenance":{"easyocr":easyocr_weight_provenance,"vintern_model_id":MODEL_ID,"vintern_revision":MODEL_REVISION,"vintern_weight_sha256":MODEL_WEIGHT_SHA256,"vintern_remote_code_git_oids":actual_code_oids},"policy_seed":{"easyocr_confidence_escalate":EASY_CONF_ESCALATE,"mixed_heuristic":"VI-marked region plus separate ASCII-only >=3-letter region","wide_crop_aspect_for_two_tiles":WIDE_CROP_ASPECT_FOR_TWO_TILES,"threshold_sweep":threshold_sweep},"resume_demo":resume_demo,"metrics":{"by_video":by_video,"craft_detect_seconds":craft_seconds,"craft_frames_per_second":len(easy_rows)/max(craft_seconds,1e-9),"easyocr_recognize_seconds":recognize_seconds,"easyocr_total_seconds":easy_seconds,"combined_frames_per_second":len(easy_rows)/easy_seconds,"total_regions":len(all_regions),"vintern_candidates":len(vintern_rows),"vintern_success":len(vintern_success),"vintern_errors":sum(row["status"]=="error" for row in vintern_rows),"vintern_inference_seconds":vintern_seconds,"vintern_candidates_per_second":len(vintern_success)/max(vintern_seconds,1e-9),"gemini_residual_candidates":len(residual_rows),"vintern_load":vintern_load,"vintern_peak":vintern_peak},"eta_single_t4_sequential":{"easyocr_hours_full_catalog":easy_hours_full,"vintern_hours_full_catalog_at_observed_escalation":vintern_hours_full,"total_hours":easy_hours_full+vintern_hours_full,"warning":"Observed dev5 extrapolation; excludes I/O variance, retries and Gemini"},"manual_accuracy":{"status":"pending","review_rows":len(review),"review_csv":str(REVIEW_CSV),"contact_sheet_archive":str(review_zip),"pass_requires":"human labels across all five videos; do not infer accuracy from model agreement"},"artifacts":{"easyocr_jsonl":str(EASY_JSONL),"vintern_candidates_jsonl":str(CANDIDATE_JSONL),"vintern_results_jsonl":str(VINTERN_JSONL),"gemini_residual_canary_jsonl":str(GEMINI_RESIDUAL_JSONL)}}
atomic_json(REPORT_JSON,report)
artifact_hashes={str(path):sha256_file(path) for path in (REPORT_JSON,EASY_JSONL,CANDIDATE_JSONL,VINTERN_JSONL,GEMINI_RESIDUAL_JSONL,REVIEW_CSV,review_zip)}
print("FINAL_REPORT",REPORT_JSON); print(json.dumps(report,ensure_ascii=False,indent=2)); print("ARTIFACT_SHA256",json.dumps(artifact_hashes,indent=2))

## Required handoff

Download `ocr_gate2_dev5_report.json`, `ocr_gate2_manual_review.csv`, `ocr_gate2_manual_review_sheets.zip`, and `ocr_gate2_gemini_residual_canary.jsonl`. Fill the four blank human-review columns in the CSV before declaring accuracy PASS. Do not call Gemini yet. Stop the Kaggle session after downloading the artifacts.